# Duality and partial duality — and why the spectrum is preserved

The LCG formalism has an exact **flux–charge duality**: swap fluxes for charges,
vertices for faces, `C` for `L`, Josephson junctions for quantum phase slips, and
`G -> -1/G`. Because it is a canonical (point) transformation, **it preserves the
Hamiltonian spectrum**.

This notebook shows two faces of it:

1. **Full dual** — `dual(circuit)` dualizes the *whole* circuit. Transmon ⇄ QPS qubit.
2. **Partial dual** — `move_across_gyrator(...)` dualizes *one reciprocal
   subcircuit* and slides it across a gyrator. This is the manuscript's
   "Partial Dual Transformations": *any* circuit in parallel/series with one
   branch of a gyrator can be moved across it by taking its dual.

In [ ]:
import numpy as np
from fluxcharge import library, dual, Circuit, move_across_gyrator

## 1. Full dual: transmon ⇄ QPS qubit

The transmon is `q^2/2C - E_J cos(phi)`. Its dual swaps charge for flux and the
junction for a phase slip: `phi^2/2C - E_J cos(2q)`. The element *values* are
preserved (`E_J`, `C` carry over), so the spectra match with the **same**
parameters — no remapping.

In [ ]:
tr = library.transmon();  rt = tr.hamiltonian()
td = dual(tr);            rd = td.hamiltonian()
print("transmon :", rt.H)
print("dual     :", rd.H)

In [ ]:
p = {"C": 1.0, "E_J": 10.0}
e_tr = rt.eigenenergies(p, n_levels=6, cutoffs={"phi_v2": 60})
e_td = rd.eigenenergies(p, n_levels=6)
e_tr = np.asarray(e_tr) - e_tr[0]
e_td = np.asarray(e_td) - e_td[0]
print("transmon gaps:", np.round(e_tr, 6))
print("dual     gaps:", np.round(e_td, 6))
print("max|diff| =", np.max(np.abs(e_tr - e_td)))
assert np.max(np.abs(e_tr - e_td)) < 1e-9     # preserved to machine precision

`dual` is an involution: dualizing twice returns the original circuit (up to a
global orientation flip), so `dual(dual(c))` has the same Hamiltonian as `c`.

In [ ]:
rtt = dual(dual(tr)).hamiltonian()
print("dual(dual(transmon)):", rtt.H)

## 2. Partial dual: move a subcircuit across a gyrator

Build a transmon block — a Josephson junction **in parallel** with a capacitor —
terminating one port of a gyrator. The partial-dual move relocates that block to
the gyrator's *other* port as its **dual**: duality swaps parallel for series, so
`JJ || C` becomes a **series** `QPS — L`, and the gyrator is consumed by the
deletion rule.

> *"any circuit configuration in parallel or in series with one branch of a
> gyrator may be moved across the gyrator by taking the dual of the subcircuit in
> question."* — manuscript, Sec. *Partial Dual Transformations*

In [ ]:
A = Circuit()
A.add_capacitor("c0", "a", "g", C="C0")        # a spectator one-port on the far side
A.add_josephson("jb", "b", "g", EJ="E_J")      # transmon block: JJ ...
A.add_capacitor("cj", "b", "g", C="C_J")       #            ... || C_J
A.add_gyrator(("e1", "a", "g"), ("e2", "b", "g"), G=1)
A.ground = "g"
rA = A.hamiltonian(strict=False, canonical=True)
print("before — elements:", sorted(type(e).__name__ for e in A._elements))
print("before — H:", rA.H)

In [ ]:
B = move_across_gyrator(A, ["jb", "cj"])       # JJ || C  ->  QPS -- L (series)
print("after  — elements:", sorted(type(e).__name__ for e in B._elements))
rB = B.hamiltonian(strict=False, canonical=True)
print("after  — H:", rB.H)

In [ ]:
# spectrum preserved across the partial-dual move
p = {"C0": 1.0, "E_J": 6.0, "C_J": 1.5}
cutA = {str(b): 41 for _a, b, _c in rA.conjugate_pairs}
cutB = {str(b): 41 for _a, b, _c in rB.conjugate_pairs}
ea = rA.eigenenergies(p, n_levels=5, cutoffs=cutA); ea = np.asarray(ea) - ea[0]
eb = rB.eigenenergies(p, n_levels=5, cutoffs=cutB); eb = np.asarray(eb) - eb[0]
print("before gaps:", np.round(ea, 4))
print("after  gaps:", np.round(eb, 4))
print("max|diff| =", np.max(np.abs(ea - eb)))
assert np.allclose(ea, eb, atol=1e-3)

### Visualizing the move

Draw the circuit before and after. The parallel block `JJ || C_J` on one gyrator
port (left) is relocated across the gyrator as a **series** `QPS — L` chain
(right) — the gyrator is consumed by the deletion rule. The phase slip is drawn
as a box (center line normal to the wire); the inductor as a coil.

In [ ]:
from fluxcharge import draw_schematic
from IPython.display import Image, display

draw_schematic(A, file="partial_dual_before.png")   # JJ||C on a gyrator port
draw_schematic(B, file="partial_dual_after.png")    # -> series QPS--L, gyrator gone
print("before:");  display(Image("partial_dual_before.png"))
print("after:");   display(Image("partial_dual_after.png"))

### Keeping the gyrator: move only *part* of a port

The move above carried the *whole* port across, so the source port emptied and
the gyrator was deleted. If instead you move only **some** of the elements on a
port, the gyrator **stays** — its far half-edge is split and the moved element
reappears, dualized, in series with the gyrator's far port. This is the
manuscript's general "continuous families of partial dual circuits": put many
elements on both ports and slide any subset across, leaving the gyrator in place.

Here a gyrator couples two LC tanks (`C_a || L_b` on one port, `C_c || L_d` on
the other). We move just `C_a` across — it becomes an inductor `L = C_a/G²` in
series on the far side — and the gyrator remains.

In [ ]:
G = Circuit()
G.add_capacitor("ca", "a", "g", C="C_a")   # near port: C_a || L_b
G.add_inductor ("lb", "a", "g", L="L_b")
G.add_capacitor("cc", "b", "g", C="C_c")   # far  port: C_c || L_d
G.add_inductor ("ld", "b", "g", L="L_d")
G.add_gyrator(("gn", "a", "g"), ("gf", "b", "g"), G="G")
G.ground = "g"
rG = G.hamiltonian(strict=False, canonical=True)
print("before — elements:", sorted(type(e).__name__ for e in G._elements))
print("before — H:", rG.H)

In [ ]:
Gp = move_across_gyrator(G, "ca")           # move ONLY C_a; leave L_b -> gyrator stays
n_gyr = sum(1 for e in Gp._elements if type(e).__name__ == "Gyrator")
print("after  — elements:", sorted(type(e).__name__ for e in Gp._elements))
print("after  — gyrator retained:", n_gyr == 1)
rGp = Gp.hamiltonian(strict=False, canonical=True)
print("after  — H:", rGp.H)

In [ ]:
# spectrum preserved exactly (a point transformation), at a concrete G != 1
p = {"C_a": 1.0, "L_b": 1.3, "C_c": 0.8, "L_d": 1.1, "G": 2.0}
cg  = {str(b): 21 for _a, b, _c in rG.conjugate_pairs}
cgp = {str(b): 21 for _a, b, _c in rGp.conjugate_pairs}
e0 = rG.eigenenergies(p, n_levels=6, cutoffs=cg);  e0 = np.asarray(e0) - e0[0]
e1 = rGp.eigenenergies(p, n_levels=6, cutoffs=cgp); e1 = np.asarray(e1) - e1[0]
print("before gaps:", np.round(e0, 5))
print("after  gaps:", np.round(e1, 5))
print("max|diff| =", np.max(np.abs(e0 - e1)))
assert np.allclose(e0, e1, atol=1e-9)

In [ ]:
from fluxcharge import draw_schematic
from IPython.display import Image, display

draw_schematic(G,  file="partial_retain_before.png")   # gyrator couples two LC tanks
draw_schematic(Gp, file="partial_retain_after.png")    # C_a slid across; gyrator remains
print("before (move only C_a):"); display(Image("partial_retain_before.png"))
print("after  (gyrator retained, C_a -> series inductor on the far port):")
display(Image("partial_retain_after.png"))

### Honesty about scope

* **Full `dual` of a *single-mode* circuit** preserves the spectrum to machine
  precision (shown above, ~1e-15).
* **Partial dual (`move_across_gyrator`)** is a point transformation and
  preserves both the spectrum and well-posedness; for a nonlinear element with
  `|G| != 1` the gyration ratio lands inside the cosine (`cos(q/G)`) and the move
  *warns* (representable as a standard element only at `|G| = 1`).
* **Full `dual` of a genuinely multi-mode circuit** (e.g. 0-π) is **not yet
  validated** — the numeric canonicalization for dense, compact multi-mode forms
  is guarded (`CompactLatticeError`) rather than silently wrong. The *symbolic*
  dual Hamiltonian is still correct.